# Семинар 6. AutoML и объяснимость модели (XAI)

**Цель семинара:** Освоить ускоренное прототипирование ML-решений с помощью систем автоматического машинного обучения (AutoML) и научиться вскрывать логику «черного ящика» сложнейших ансамблей методами объяснимого ИИ (SHAP). В корпоративном секторе модель без обоснования прогноза не пройдет комплаенс, поэтому мы научимся генерировать глобальные и локальные интерпретации.

### 🔧 Настройка окружения и импорт библиотек

Для этого семинара вам понадобятся библиотеки `pycaret` (для AutoML) и `shap` (для XAI). Убедитесь, что они установлены в вашем окружении (например, `uv add pycaret shap`).


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

try:
    from pycaret.classification import setup, compare_models, pull, finalize_model, save_model
    import shap
except ImportError:
    print("⚠️ Установите пакеты: uv add pycaret shap")

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


---

## 📥 Шаг 1. Инициализация локального контура (Витрина АВТ)

Алгоритмы AutoML обучаются на полностью собранной и очищенной широкой витрине `abt_result.csv`, которую мы подготовили на прошлом семинаре.


In [ ]:
BASE_DATA_DIR = os.path.join(".", "data")
INPUT_DIR = os.path.join(BASE_DATA_DIR, "seminar_5_ml_modeling")
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, "seminar_6_automl_shap")
os.makedirs(OUTPUT_DIR, exist_ok=True)

abt_csv_path = os.path.abspath(os.path.join(INPUT_DIR, "abt_result.csv"))

if not os.path.exists(abt_csv_path):
    print(f"⚠️ Витрина не найдена: {abt_csv_path}. Генерируется mock-датасет для AutoML.")
    os.makedirs(INPUT_DIR, exist_ok=True)
    
    # Генерация синтетической витрины (500 строк для стабильной работы AutoML)
    np.random.seed(42)
    n = 500
    mock_abt = pd.DataFrame({
        'Target_ID': [f'T{i}' for i in range(n)],
        'Target_Flag': np.random.choice([0, 1], n, p=[0.75, 0.25]),
        'tenure': np.random.randint(0, 72, n),
        'TotalCharges': np.random.uniform(50, 5000, n),
        'Recency': np.random.randint(1, 100, n),
        'Monetary': np.random.uniform(10, 2000, n),
        'Mean_Sentiment': np.random.uniform(-1.0, 1.0, n),
        'Contract_one_year': np.random.choice([0, 1], n)
    })
    # Добавляем немного логики для SHAP: чем больше сентимент, тем меньше шанс оттока (1)
    mock_abt.loc[mock_abt['Mean_Sentiment'] > 0.5, 'Target_Flag'] = 0
    mock_abt.to_csv(abt_csv_path, index=False)

df_abt = pd.read_csv(abt_csv_path)
print(f"Витрина успешно загружена. Размерность: {df_abt.shape}")


---

## 🛠 ЗАДАНИЕ 1: Конфигурация AutoML и поиск лучшей модели
**Бизнес-контекст:** Вместо ручного перебора десятков архитектур (Деревья, Регрессии, Бустинги) и их гиперпараметров, бизнес использует AutoML. Это экономит сотни часов Data Science команды. Система сама проводит k-fold кросс-валидацию и возвращает лидерборд алгоритмов.

**Инструкция (TODO):**
1. Инициализируйте пайплайн `setup()` из `pycaret.classification`. Передайте `data=df_abt`, `target='Target_Flag'`, `ignore_features=['Target_ID']`, и `session_id=42` для воспроизводимости.
2. Запустите автоматический поиск лучшей модели с помощью `compare_models()`.
3. Сохраните таблицу результатов кросс-валидации с помощью функции `pull()`.

*🤖 Теги для AI-ментора: `#SEM6_TASK1_START`, `#SEM6_TASK1_BUG`*


In [ ]:
# [MASTER SOLUTION]
print("⏳ Запуск AutoML пайплайна (может занять 1-2 минуты)...")

# 1. Инициализация (silent=True отключает интерактивный ввод)
target_col = [c for c in ['Churn', 'Target_Flag', 'Exited', 'Default'] if c in df_abt.columns][0]

clf_setup = setup(
    data=df_abt, 
    target=target_col, 
    ignore_features=['Target_ID'],
    session_id=42,
    verbose=False
)

# 2. Поиск лучшей модели (в учебных целях ограничим список для скорости)
best_model = compare_models(include=['lr', 'dt', 'rf', 'xgboost', 'lightgbm'])

# 3. Получение лидерборда
leaderboard = pull()
print("\n🏆 Топ-3 модели по кросс-валидации:")
display(leaderboard.head(3))

print(f"\n✅ Выбранная лучшая модель: {type(best_model).__name__}")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1.1. Вызовите функцию setup(...) передав df_abt, имя таргета и игнорируя Target_ID
# TODO: 1.2. Запустите compare_models() и сохраните в best_model
# TODO: 1.3. Вызовите pull(), чтобы получить DataFrame с метриками и выведите его через display()
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

target_col = [c for c in ['Churn', 'Target_Flag', 'Exited', 'Default'] if c in df_abt.columns][0]

clf_setup = setup(
    data=df_abt, 
    target=target_col, 
    ignore_features=['Target_ID'],
    session_id=42,
    verbose=False
)

best_model = compare_models(...)
leaderboard = ...
display(leaderboard.head(3))


---

## 🛠 ЗАДАНИЕ 2: Глобальный аудит рисков (SHAP Summary Plot)
**Бизнес-контекст:** Топ-менеджменту не нужны метрики `Accuracy`. Им нужно знать, **какие макро-факторы** заставляют клиентов уходить. Основываясь на теории кооперативных игр (векторы Шепли), `SHAP` вычисляет честный вес каждого признака.

**Инструкция (TODO):**
1. Извлеките матрицу признаков, на которой обучалась модель (в PyCaret это `clf_setup.X_train_transformed` или просто подготовьте `X_train` руками).
2. Инициализируйте `shap.TreeExplainer(best_model)` (Ансамбли на основе деревьев отлично работают с `TreeExplainer`).
3. Вычислите SHAP-значения: `shap_values = explainer.shap_values(X_train)`.
4. Постройте график глобальной важности: `shap.summary_plot(shap_values, X_train)`.

*🤖 Теги для AI-ментора: `#SEM6_TASK2_START`, `#SEM6_TASK2_WHY`*


In [ ]:
# [MASTER SOLUTION]
print("⏳ Генерация глобальной XAI интерпретации...")

# Получаем данные после обработки AutoML
try:
    X_train = clf_setup.dataset.drop(columns=['Target_Flag', 'Target_ID'])
except AttributeError:
    # Запасной вариант (зависит от версии PyCaret)
    X_train = df_abt.drop(columns=['Target_Flag', 'Target_ID'])

# Инициализируем SHAP (используем TreeExplainer для ансамблей)
explainer = shap.TreeExplainer(best_model)

# В некоторых ансамблях (RF) shap_values возвращает список (для каждого класса)
# Берем индекс [1] для положительного класса (отток/дефолт)
shap_values = explainer.shap_values(X_train)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

# Визуализация
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_train, show=False)
plt.title("SHAP Глобальная важность признаков (Summary Plot)")
plt.show()

# Вывод: Признаки вверху графика оказывают наибольшее влияние на прогноз. 
# Красный цвет означает высокое значение признака, синий - низкое.


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 2.1. Подготовьте X_train (удалите таргет и ID)
# TODO: 2.2. Инициализируйте shap.TreeExplainer, передав best_model
# TODO: 2.3. Рассчитайте explainer.shap_values(X_train)
# TODO: 2.4. Постройте shap.summary_plot
raise NotImplementedError("Задание 2 не выполнено! Удалите эту строку и напишите свой код.")

X_train = df_abt.drop(columns=[...])

explainer = shap.TreeExplainer(...)
shap_values = explainer.shap_values(...)

# Защита от мультиклассового вывода для деревьев
if isinstance(shap_values, list):
    shap_values = shap_values[1]

plt.figure(figsize=(10, 6))
shap.summary_plot(..., ..., show=False)
plt.show()


---

## 🛠 ЗАДАНИЕ 3: Локальное обоснование ИИ-решения (SHAP Waterfall / Force Plot)
**Бизнес-контекст:** Линейному менеджеру (фронт-офис) предстоит звонить клиенту. Почему именно этому? Алгоритм счел его "в зоне риска". Локальное объяснение покажет, **какие именно триггеры** в истории этого конкретного человека склонили чашу весов.

**Инструкция (TODO):**
1. Выберите одного клиента из базы (например, строку с индексом `idx = 0`).
2. Извлеките его `shap_values` и базовое значение модели (`explainer.expected_value`).
3. Постройте локальный график: `shap.force_plot(...)` (В Jupyter требует `shap.initjs()`) или `shap.waterfall_plot(...)`. Мы используем классический `bar` для совместимости.


In [ ]:
# [MASTER SOLUTION]
# Возьмем нулевого клиента
idx = 0
client_data = X_train.iloc[idx]
client_shap_values = shap_values[idx]
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = expected_value[1]

print(f"--- Разбор прогноза для клиента ---")
print(f"Базовая вероятность (Expected): {expected_value:.2f}")

# Создадим объект Explanation для удобного вывода waterfall/bar (новые версии SHAP)
try:
    explanation = shap.Explanation(
        values=client_shap_values, 
        base_values=expected_value, 
        data=client_data, 
        feature_names=X_train.columns
    )
    plt.figure(figsize=(8, 4))
    shap.plots.waterfall(explanation, show=False)
    plt.title(f"Локальное обоснование (SHAP Waterfall) для Клиента #{idx}")
    plt.show()
except Exception:
    # Запасной вариант для старых версий
    shap.summary_plot(client_shap_values.reshape(1,-1), client_data.to_frame().T, plot_type="bar", show=False)
    plt.show()


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 3.1. Выберите индекс клиента (например, 0)
# TODO: 3.2. Извлеките client_data = X_train.iloc[idx] и client_shap_values = shap_values[idx]
# TODO: 3.3. Визуализируйте локальный вклад (используйте код из решения выше)
raise NotImplementedError("Задание 3 не выполнено! Удалите эту строку и напишите свой код.")

idx = ...
client_data = X_train.iloc[...]
client_shap_values = shap_values[...]
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value = expected_value[1]

explanation = shap.Explanation(
    values=..., 
    base_values=..., 
    data=..., 
    feature_names=X_train.columns
)
shap.plots.waterfall(explanation)


---

## 🏗 ФИНАЛЬНАЯ СБОРКА: Сквозная функция run_automl_and_explain

Для `src/model_training.py` мы напишем функцию-оркестратор `run_automl_and_explain`. Она примет собранную АВТ-витрину, запустит AutoML, оттюнингует лучшую модель, сгенерирует объекты SHAP и вернет кортеж `(model, shap_explainer, metrics)`.


In [ ]:
# [MASTER SOLUTION]
def run_automl_and_explain(df: pd.DataFrame, target_col: str, task_type: str = 'classification') -> tuple:
    """
    Промышленная функция AutoML оптимизации и инициализации ХАI.
    Возвращает финализированную модель, explainer и таблицу метрик.
    """
    print("🚀 Старт модуля AutoML & XAI...")
    
    # 1. Инициализация среды PyCaret
    clf_setup = setup(
        data=df,
        target=target_col,
        ignore_features=['Target_ID'] if 'Target_ID' in df.columns else None,
        session_id=42,
        verbose=False,
        html=False
    )
    
    # 2. Поиск лучшей модели (в проде список можно убрать для перебора всех)
    best_model = compare_models(include=['lr', 'dt', 'rf', 'lightgbm'], verbose=False)
    leaderboard = pull()
    
    # 3. Финализация модели на всех данных
    final_model = finalize_model(best_model)
    
    # 4. Инициализация SHAP
    X_train = df.drop(columns=[target_col, 'Target_ID'], errors='ignore')
    
    # Подбор Explainer'а в зависимости от архитектуры (Деревья vs Линейные)
    try:
        explainer = shap.TreeExplainer(final_model)
        # Тестовый прогон для валидации
        _ = explainer.shap_values(X_train.iloc[:5]) 
    except Exception:
        # Fallback для линейных моделей и прочих
        # В PyCaret final_model это Pipeline, берем последний шаг
        actual_model = final_model.steps[-1][1] if hasattr(final_model, 'steps') else final_model
        try:
             explainer = shap.TreeExplainer(actual_model)
        except Exception:
             explainer = shap.Explainer(actual_model, X_train)
    
    # Сохранение артефакта модели на диск для Семинара 8
    os.makedirs('models', exist_ok=True)
    save_model(final_model, 'models/model')
    
    return final_model, explainer, leaderboard

# Запуск пайплайна
final_model, explainer, metrics = run_automl_and_explain(df_abt, target_col='Target_Flag')
print("Пайплайн завершен. Топ-1 Модель:", type(final_model.steps[-1][1] if hasattr(final_model, 'steps') else final_model).__name__)


In [ ]:
# [STUDENT TEMPLATE]
def run_automl_and_explain(df: pd.DataFrame, target_col: str, task_type: str = 'classification') -> tuple:
    # TODO: Реализуйте setup(), compare_models(), finalize_model()
    # TODO: Инициализируйте shap.TreeExplainer
    # TODO: Верните кортеж (final_model, explainer, leaderboard)
    raise NotImplementedError("Финальная сборка функции не выполнена!")


---

## 🛠 Автоматизированная проверка качества (Autocheck)

Скрипт проверяет работоспособность кортежа возврата и сохранение эталонного артефакта модели `model.pkl`.


In [ ]:
def run_autocheck(model_tuple):
    print("🚀 Тестирование функции run_automl_and_explain...\n" + "-"*45)
    validation_status = True
    
    if not isinstance(model_tuple, tuple) or len(model_tuple) != 3:
        print("❌ Ошибка: Функция должна возвращать кортеж из 3 элементов (model, explainer, leaderboard).")
        return False
        
    final_model, explainer, metrics = model_tuple
    
    # Проверка модели
    if not hasattr(final_model, 'predict'):
        print("❌ Ошибка: Первый элемент не является обученной моделью (отсутствует метод predict).")
        validation_status = False
    else:
        print("✅ Модель успешно обучена и финализирована.")
        
    # Проверка Explainer'а
    if not hasattr(explainer, 'shap_values'):
        print("❌ Ошибка: Второй элемент не является валидным объектом shap.Explainer.")
        validation_status = False
    else:
        print("✅ SHAP Explainer (XAI) успешно инициализирован.")
        
    # Проверка метрик
    if not isinstance(metrics, pd.DataFrame) or metrics.empty:
        print("❌ Ошибка: Третий элемент (Leaderboard) не является DataFrame или пуст.")
        validation_status = False
    else:
        print("✅ Лидерборд AutoML корректно сгенерирован.")
        
    # Проверка экспорта (расширение .pkl добавляется PyCaret автоматически)
    if os.path.exists('models/model.pkl'):
         print("✅ Эталонный артефакт model.pkl успешно сериализован в директорию models/.")
    else:
         print("❌ Ошибка: Файл model.pkl не найден. Проверьте вызов save_model().")
         validation_status = False

    print("-" * 45)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! Пайплайн AutoML & XAI готов.")
        print("Перенесите эту функцию в course_project/src/model_training.py!")
    else:
        print("⚠️ Обнаружены дефекты.")

run_autocheck((final_model, explainer, metrics))
